In [ ]:
import os

import numpy as np
from barrier3d import Barrier3d
from matplotlib import pyplot as plt

from cascade.outwasher import Outwasher

# Outwash every 10 years

In [ ]:
b3d = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
years = np.load(
    "C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years10.npy"
)

for t in range(1, 101):
    b3d.update()
    b3d.update_dune_domain()
    if b3d._time_index - 1 in years:
        outwash = Outwasher(
            datadir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/",
            outwash_years="outwash_years10.npy",
            outwash_bay_levels="outwash_baylevels10.npy",
            time_step_count=b3d._TMAX,
            berm_elev=b3d._BermEl,
            barrier_length=b3d._BarrierLength,
            sea_level=b3d._SL,
            bay_depth=-b3d._BayDepth,
            interior_domain=b3d.InteriorDomain,
            dune_domain=b3d.DuneDomain[b3d._time_index - 1],
            block_size=5,
            substep=20,
            sediment_flux_coefficient_Cx=10,
            sediment_flux_coefficient_Ki=7.5e-3,  # b3d = 7.5E-6 for inundation
            max_slope=-0.25,
            shoreface_on=False,
        )
        outwash.update(b3d)

    print("B3D time step: ", t)

In [ ]:
b3d_domain = b3d.InteriorDomain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
# fig1, (ax1, ax3) = plt.subplots(1, 2, sharey=True)
mat = ax1.matshow(
    b3d_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()
# fig1.savefig(newpath + "0_domain", facecolor='w')

In [ ]:
path = "D:/NC State/Outwasher/Output/newest_flow_routing/"
runID = "100years_outwash10_substep_20_shoreface_off"
newpath = path + runID + "/"
if not os.path.exists(newpath):
    os.makedirs(newpath)
os.chdir(newpath)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

In [ ]:
outwash._Qs_shoreface[b3d._time_index - 1]  # m^3

In [ ]:
import os

initial_domain = outwash._initial_full_domain
final_domain = outwash._full_domain
domain_change = final_domain - initial_domain

### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
# fig1, (ax1, ax3) = plt.subplots(1, 2, sharey=True)
mat = ax1.matshow(
    initial_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()
fig1.savefig(newpath + "0_domain", facecolor="w")

# plotting post storm elevation
fig2 = plt.figure()
ax2 = fig2.add_subplot(111)
mat2 = ax2.matshow(
    final_domain[:, :] * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
ax2.set_xlabel("barrier length (dam)")
ax2.set_ylabel("barrier width (dam)")
ax2.set_title("Final Elevation")
plt.gca().xaxis.tick_bottom()
cbar = fig2.colorbar(mat2)
cbar.set_label("m MHW", rotation=270, labelpad=15)
fig2.savefig(newpath + "final_domain", facecolor="w")

# plotting domain elevation change
domain_change = final_domain - initial_domain
fig3 = plt.figure()
ax3 = fig3.add_subplot(111)
mat3 = ax3.matshow(
    domain_change * 10,
    cmap="seismic",
    vmin=-2,
    vmax=2,
)
ax3.set_xlabel("barrier length (dam)")
ax3.set_ylabel("barrier width (dam)")
ax3.set_title("Elevation Change")
plt.gca().xaxis.tick_bottom()
cbar = fig3.colorbar(mat3)
cbar.set_label("meters", rotation=270, labelpad=15)
fig3.savefig(newpath + "elev_change_domain", facecolor="w")

In [ ]:
### HYDROGRAPH------------------------------------------------------------------------------------------------------------------
# plot the bay elevation throughout each storm with sea level and beach elevation references
# plt.rcParams['figure.figsize'] = (8,5)

bay_levels = outwash._final_bay_levels
duration = len(bay_levels)
x = range(0, duration)
beach_elev_line = outwash._beach_elev * np.ones(len(x))
dune_elev_line = max(outwash._full_dunes[0]) * np.ones(len(x))

fig5 = plt.figure()
ax5 = fig5.add_subplot(111)
ax5.plot(x, bay_levels * 10, label="hydrograph")
# if we have multiple storms, will only need to plot these once
ax5.plot(x, beach_elev_line * 10, "k", linestyle="dotted", label="berm elevation")
ax5.plot(x, dune_elev_line * 10, "k", linestyle="dashdot", label="max dune elevation")
ax5.set_xlabel("storm duration")
ax5.set_ylabel("m MHW")
ax5.set_title("bay elevation over the course of each storm")
ax5.legend(loc="lower right")
plt.show()
fig5.savefig(newpath + "hydrograph", facecolor="w")
plt.close()

In [ ]:
# from scripts.outwash_ms.plotters_outwash import (plot_ElevAnimation, plot_DischargeAnimation, plot_FRarray)

In [ ]:
# plt.rcParams.update({"font.size": 15})
# start = outwash._OW_TS[0]
# # stop = outwash._OW_TS[-1]
# stop = start+200
# # path = "D:/NC State/Outwasher/Output/full_hydro/"
# # runID = "Kie-3_substep20_B3D63_extra_row_C0_10x10"
# # newpath = path + runID + "/"
# # plot_ElevAnimation(elev_change, newpath, start, stop)
# plot_DischargeAnimation(outwash._discharge[b3d._time_index-1], newpath, start, stop)
# plot_FRarray(outwash._flow_routing_cellular_array[b3d._time_index-1], newpath, start, stop)

In [ ]:
b3d.x_s_TS[-1]  # dam

In [ ]:
b3d.x_s_TS[-2]  # dam

In [ ]:
m_xsTS = np.multiply(b3d.x_s_TS, 10)
plt.plot(m_xsTS)
plt.vlines(
    years - 1, ymin=min(m_xsTS), ymax=max(m_xsTS), colors="k", linestyles="dotted"
)
plt.xlabel("time")
plt.ylabel("shoreline position")
plt.savefig(newpath + "shoreline_position", facecolor="w")

# B3D Only

In [ ]:
b3d2 = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
for t in range(1, 101):
    b3d2.update()
    b3d2.update_dune_domain()
    print("B3D time step: ", t)

In [ ]:
# path = "D:/NC State/Outwasher/Output/newest_flow_routing/"
# runID = "100years_B3D"
# newpath = path + runID + "/"
# if not os.path.exists(newpath):
#     os.makedirs(newpath)
# os.chdir(newpath)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})
m_xsTS2 = np.multiply(b3d2.x_s_TS, 10)
plt.plot(m_xsTS2, label="B3D only")
plt.plot(m_xsTS, label="Outwash every 10 years")
# plt.vlines(years-1, ymin=min(m_xsTS), ymax=580, colors='k', linestyles='dotted')
plt.xlabel("time")
plt.ylabel("shoreline position")
plt.ylim([500, 580])
plt.legend()
plt.title("Outwash to Offshore")
plt.savefig(
    r"D:\NC State\Outwasher\Output\newest_flow_routing/b3d_outwash10_shorelinechange_SFoff",
    facecolor="w",
)

# Outwash every 20 years

In [ ]:
b3d3 = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/Barrier3d/tests/test_params/")
years3 = np.load(
    "C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years20.npy"
)

for t in range(1, 101):
    b3d3.update()
    b3d3.update_dune_domain()
    if b3d3._time_index - 1 in years3:
        outwash3 = Outwasher(
            datadir="C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/",
            outwash_years="outwash_years20.npy",
            outwash_bay_levels="outwash_baylevels20.npy",
            time_step_count=b3d3._TMAX,
            berm_elev=b3d3._BermEl,
            barrier_length=b3d3._BarrierLength,
            sea_level=b3d3._SL,
            bay_depth=-b3d3._BayDepth,
            interior_domain=b3d3.InteriorDomain,
            dune_domain=b3d3.DuneDomain[b3d3._time_index - 1],
            block_size=5,
            substep=20,
            sediment_flux_coefficient_Cx=10,
            sediment_flux_coefficient_Ki=7.5e-3,  # b3d = 7.5E-6 for inundation
            max_slope=-0.25,
            shoreface_on=False,
        )
        outwash.update(b3d3)

    print("B3D time step: ", t)

In [ ]:
path = "D:/NC State/Outwasher/Output/newest_flow_routing/"
runID = "100years_outwash20_substep_20_shoreface_off"
newpath = path + runID + "/"
if not os.path.exists(newpath):
    os.makedirs(newpath)
os.chdir(newpath)

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

In [ ]:
import os

initial_domain = outwash._initial_full_domain
final_domain = outwash._full_domain
domain_change = final_domain - initial_domain

### ELEVATION PLOTS-------------------------------------------------------------------------------------------------------------
# plotting initial domain
fig1 = plt.figure()
ax1 = fig1.add_subplot(111)
# fig1, (ax1, ax3) = plt.subplots(1, 2, sharey=True)
mat = ax1.matshow(
    initial_domain * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
cbar = fig1.colorbar(mat)
cbar.set_label("m MHW", rotation=270, labelpad=15)
ax1.set_title("Initial Elevation")
ax1.set_ylabel("barrier width (dam)")
ax1.set_xlabel("barrier length (dam)")
plt.gca().xaxis.tick_bottom()
fig1.savefig(newpath + "0_domain", facecolor="w")

# plotting post storm elevation
fig2 = plt.figure()
ax2 = fig2.add_subplot(111)
mat2 = ax2.matshow(
    final_domain[:, :] * 10,
    cmap="terrain",
    vmin=-3.0,
    vmax=3.0,
)
ax2.set_xlabel("barrier length (dam)")
ax2.set_ylabel("barrier width (dam)")
ax2.set_title("Final Elevation")
plt.gca().xaxis.tick_bottom()
cbar = fig2.colorbar(mat2)
cbar.set_label("m MHW", rotation=270, labelpad=15)
fig2.savefig(newpath + "final_domain", facecolor="w")

# plotting domain elevation change
domain_change = final_domain - initial_domain
fig3 = plt.figure()
ax3 = fig3.add_subplot(111)
mat3 = ax3.matshow(
    domain_change * 10,
    cmap="seismic",
    vmin=-2,
    vmax=2,
)
ax3.set_xlabel("barrier length (dam)")
ax3.set_ylabel("barrier width (dam)")
ax3.set_title("Elevation Change")
plt.gca().xaxis.tick_bottom()
cbar = fig3.colorbar(mat3)
cbar.set_label("meters", rotation=270, labelpad=15)
fig3.savefig(newpath + "elev_change_domain", facecolor="w")

# Comparison Figures

In [ ]:
m_xsTS3 = np.multiply(b3d3.x_s_TS, 10)
plt.vlines(
    years3 - 1, ymin=min(m_xsTS3), ymax=max(m_xsTS3), colors="r", linestyles="dotted"
)
plt.xlabel("time")
plt.ylabel("shoreline position")
m_xsTS2 = np.multiply(b3d2.x_s_TS, 10)
plt.plot(m_xsTS2, label="B3D only")
plt.plot(m_xsTS, label="Outwash every 10 years")
plt.plot(m_xsTS3, label="Outwash every 20 years")
plt.vlines(
    years - 1, ymin=min(m_xsTS), ymax=max(m_xsTS), colors="k", linestyles="dotted"
)
plt.xlabel("time")
plt.ylabel("shoreline position")
# plt.savefig(newpath + "shoreline_position", facecolor='w')
plt.legend()
plt.savefig(
    r"D:\NC State\Outwasher\Output\newest_flow_routing\shoreface_nourishment_comparison_shoreface_off_substep20.png",
    facecolor="w",
)

In [ ]:
OWTS = b3d.QowTS  # m3/m
OWTS2 = b3d2.QowTS  # m3/m
OWTS3 = b3d3.QowTS  # m3/m

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams.update({"font.size": 15})

plt.plot(OWTS2, label="B3D only")
plt.plot(OWTS, linestyle="dashed", label="Outwash every 10 years")
# plt.plot(OWTS3, label="Outwash every 20 years")
# plt.vlines(years3-1, ymin=min(m_xsTS3), ymax=max(m_xsTS3), colors='r', linestyles='dotted')
# plt.vlines(years-1, ymin=min(m_xsTS), ymax=max(m_xsTS), colors='k', linestyles='dotted')
plt.xlabel("time")
plt.ylabel("overwash flux [$m^3/m$]")
# plt.savefig(newpath + "shoreline_position", facecolor='w')
plt.legend()
plt.title("Outwash to Offshore")
plt.savefig(
    r"D:\NC State\Outwasher\Output\newest_flow_routing\overwash_comparison_substep20.png",
    facecolor="w",
)

In [ ]:
sfTS = b3d.s_sf_TS  # m3/m
sfTS2 = b3d2.s_sf_TS  # m3/m
sfTS3 = b3d3.s_sf_TS  # m3/m

plt.plot(sfTS2, label="B3D only")
plt.plot(sfTS, label="Outwash every 10 years")
plt.plot(sfTS3, label="Outwash every 20 years")

plt.xlabel("time")
plt.ylabel("shoreface slope")
# plt.savefig(newpath + "shoreline_position", facecolor='w')
plt.legend()
# plt.savefig(r"D:\NC State\Outwasher\Output\newest_flow_routing\overwash_comparison_substep20.png", facecolor='w')

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

dune_min1 = []
dune_max1 = []
dune_avg1 = []
dune_min2 = []
dune_max2 = []
dune_avg2 = []
dune_min3 = []
dune_max3 = []
dune_avg3 = []

for t in range(101):
    # b3d only
    sub_domain2 = b3d2._DuneDomain[t, :, :]
    crest2 = sub_domain2.max(axis=1)
    dune_min2.append(min(crest2))
    dune_max2.append(max(crest2))
    dune_avg2.append(np.mean(crest2))
    # outwash 10 years
    sub_domain1 = b3d._DuneDomain[t, :, :]
    crest1 = sub_domain1.max(axis=1)
    dune_min1.append(min(crest1))
    dune_max1.append(max(crest1))
    dune_avg1.append(np.mean(crest1))
    # outwash 20 years
    sub_domain3 = b3d3._DuneDomain[t, :, :]
    crest3 = sub_domain3.max(axis=1)
    dune_min3.append(min(crest3))
    dune_max3.append(max(crest3))
    dune_avg3.append(np.mean(crest3))

plt.figure(1)
plt.plot(dune_min2, label="B3D Only")
plt.plot(dune_min1, label="Outwash every 10 years")
plt.plot(dune_min3, label="Outwash every 20 years")
plt.title("min dune crest heights")
plt.xlabel("time")
plt.legend()

plt.figure(2)
plt.plot(dune_max2, label="B3D Only")
plt.plot(dune_max1, label="Outwash every 10 years")
plt.plot(dune_max3, label="Outwash every 20 years")
plt.title("max dune crest heights")
plt.xlabel("time")
plt.legend()

plt.figure(3)
plt.plot(dune_avg2, label="B3D Only")
plt.plot(dune_avg1, label="Outwash every 10 years")
plt.plot(dune_avg3, label="Outwash every 20 years")
plt.title("avg dune crest heights")
plt.xlabel("time")
plt.legend(loc="upper right")

In [ ]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams.update({"font.size": 15})

dune_min1 = []
dune_max1 = []
dune_avg1 = []
dune_min2 = []
dune_max2 = []
dune_avg2 = []
dune_min3 = []
dune_max3 = []
dune_avg3 = []

for t in range(101):
    # b3d only
    sub_domain2 = b3d2._DuneDomain[t, :, :]
    crest2 = sub_domain2[:, 0]
    dune_min2.append(min(crest2))
    dune_max2.append(max(crest2))
    dune_avg2.append(np.mean(crest2))
    # outwash 10 years
    sub_domain1 = b3d._DuneDomain[t, :, :]
    crest1 = sub_domain1[:, 0]
    dune_min1.append(min(crest1))
    dune_max1.append(max(crest1))
    dune_avg1.append(np.mean(crest1))
    # outwash 20 years
    sub_domain3 = b3d3._DuneDomain[t, :, :]
    crest3 = sub_domain3[:, 0]
    dune_min3.append(min(crest3))
    dune_max3.append(max(crest3))
    dune_avg3.append(np.mean(crest3))

plt.figure(1)
plt.plot(dune_min2, label="B3D Only")
plt.plot(dune_min1, label="Outwash every 10 years")
plt.plot(dune_min3, label="Outwash every 20 years")
plt.title("min dune heights")
plt.xlabel("time")
plt.legend(loc="upper right")

plt.figure(2)
plt.plot(dune_max2, label="B3D Only")
plt.plot(dune_max1, label="Outwash every 10 years")
plt.plot(dune_max3, label="Outwash every 20 years")
plt.title("max dune heights")
plt.xlabel("time")
plt.legend(loc="upper right")

plt.figure(3)
plt.plot(dune_avg2, label="B3D Only")
plt.plot(dune_avg1, label="Outwash every 10 years")
plt.plot(dune_avg3, label="Outwash every 20 years")
plt.title("avg dune heights")
plt.xlabel("time")
plt.legend(loc="upper right")